# Colormap Modification for Radar COGs

This notebook demonstrates how to:

1. Create a **raw float COG** from synthetic radar-like data — this preserves the original
   scientific values and stores colormap metadata for later use.
2. **Visualise PNG tiles** from the COG at different overview levels using `rasterio`.
3. **Change the colormap** of an already-created COG without reprocessing the raw radar
   data by calling `remap_cog_colormap`.
4. Read tiles on-the-fly with `read_cog_tile_as_rgba` so you can serve differently-coloured
   PNG tiles directly from an API endpoint.

> **No actual radar files are needed** — all data in this notebook is synthetic.

## 1. Setup — imports and synthetic radar data

In [ ]:
from pathlib import Path
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from PIL import Image

# Import radarlib — registers custom colormaps automatically
import radarlib  # noqa: F401
from radarlib.radar_grid import (
    GridGeometry,
    create_cog,
    create_raw_cog,
    read_cog_metadata,
    read_cog_tile_as_rgba,
    remap_cog_colormap,
)

# All output files go into a temporary directory
WORK_DIR = Path(tempfile.mkdtemp(prefix="radar_cog_demo_"))
print(f"Working directory: {WORK_DIR}")

### 1.1 Build a synthetic radar-like 2-D data array

We create a circular reflectivity pattern similar to a plan-position indicator (PPI) scan.
Values range from ≈ 0 to 70 dBZ, with NaN values outside a 140 km range.

In [ ]:
NY, NX = 300, 300           # grid cells
RANGE_M = 150_000           # ± 150 km

x = np.linspace(-RANGE_M, RANGE_M, NX)
y = np.linspace(-RANGE_M, RANGE_M, NY)
X, Y = np.meshgrid(x, y)
R = np.sqrt(X**2 + Y**2)
THETA = np.arctan2(Y, X)

# Reflectivity: decreases with range, varies with azimuth
dbzh = (
    20
    + 30 * np.exp(-R / 80_000)
    + 15 * np.exp(-((R - 50_000) / 20_000) ** 2) * (0.5 + 0.5 * np.cos(3 * THETA))
    + np.random.default_rng(0).normal(0, 2, (NY, NX))
)
dbzh = np.clip(dbzh, 0, 75).astype(np.float32)
dbzh[R > 140_000] = np.nan  # mask outside radar coverage

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(dbzh, origin="lower", cmap="viridis", vmin=0, vmax=70)
plt.colorbar(im, ax=ax, label="Reflectivity (dBZ)")
ax.set_title("Synthetic CAPPI — reflectivity (dBZ)")
plt.tight_layout()
plt.show()
print(f"Data shape: {dbzh.shape}, range: [{np.nanmin(dbzh):.1f}, {np.nanmax(dbzh):.1f}] dBZ")

### 1.2 Create a minimal GridGeometry

In [ ]:
NZ = 5   # not used for 2-D product, but required by GridGeometry
N_PTS = NZ * NY * NX

geometry = GridGeometry(
    grid_shape=(NZ, NY, NX),
    grid_limits=((0.0, 15000.0), (-RANGE_M, RANGE_M), (-RANGE_M, RANGE_M)),
    indptr=np.arange(N_PTS + 1, dtype=np.int32),
    gate_indices=np.zeros(N_PTS, dtype=np.int32),
    weights=np.ones(N_PTS, dtype=np.float32),
    toa=15000.0,
    radar_altitude=500.0,
)
print(geometry)

## 2. Create a Raw Float COG

`create_raw_cog` stores the scientific float values together with
default colormap / vmin / vmax metadata.  This is the key file that allows
you to re-render with any colormap later.

In [ ]:
raw_cog_path = WORK_DIR / "reflectivity_raw.cog"

create_raw_cog(
    dbzh,
    geometry,
    radar_lat=-31.441,   # fictitious radar location
    radar_lon=-64.191,
    output_path=raw_cog_path,
    cmap="viridis",      # default colormap stored as metadata
    vmin=0.0,
    vmax=70.0,
    projection="EPSG:3857",
    overview_factors=[2, 4, 8],
    resampling_method="nearest",
)

print(f"Raw COG written to: {raw_cog_path}")
print(f"File size: {raw_cog_path.stat().st_size / 1024:.1f} KB")

### 2.1 Inspect the stored metadata

In [ ]:
meta = read_cog_metadata(raw_cog_path)
print("Metadata stored in the raw COG:")
for k, v in meta.items():
    print(f"  {k:12s}: {v}")

### 2.2 Verify the raw values are preserved

In [ ]:
with rasterio.open(raw_cog_path) as src:
    stored = src.read(1)
    print(f"Band dtype  : {src.dtypes[0]}")
    print(f"Band count  : {src.count}")
    print(f"CRS         : {src.crs}")
    print(f"Nodata      : {src.nodata}")
    print(f"Stored range: [{np.nanmin(stored):.2f}, {np.nanmax(stored):.2f}] dBZ")

print()
print("The file preserves the original float values — no colormap has been applied yet.")

## 3. Read Tiles as PNG Images Using `read_cog_tile_as_rgba`

`read_cog_tile_as_rgba` applies any colormap on-the-fly when reading a raw COG.
This means **no file modification is required** to change how the data looks.

In [ ]:
colormaps_to_show = [
    ("viridis",     0, 70),
    ("plasma",      0, 70),
    ("hot",         0, 70),
    ("jet",         0, 70),
    ("grc_th",      0, 70),   # radarlib custom colormap
]

fig, axes = plt.subplots(1, len(colormaps_to_show), figsize=(16, 4))
for ax, (cmap_name, vmin, vmax) in zip(axes, colormaps_to_show):
    rgba = read_cog_tile_as_rgba(raw_cog_path, cmap=cmap_name, vmin=vmin, vmax=vmax)
    ax.imshow(rgba, origin="upper")
    ax.set_title(cmap_name, fontsize=10)
    ax.axis("off")

plt.suptitle("Same raw COG rendered with different colormaps", fontsize=13, weight="bold")
plt.tight_layout()
plt.show()

### 3.1 Reading at different overview levels (zoom levels)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for lvl, ax in enumerate(axes):
    rgba = read_cog_tile_as_rgba(raw_cog_path, cmap="grc_th", vmin=0, vmax=70, overview_level=lvl)
    ax.imshow(rgba, origin="upper")
    ax.set_title(f"Overview level {lvl}\n({rgba.shape[1]}×{rgba.shape[0]} px)", fontsize=9)
    ax.axis("off")

plt.suptitle("Same data at decreasing resolutions (overview pyramid)", fontsize=12, weight="bold")
plt.tight_layout()
plt.show()

### 3.2 Reading a spatial window (tile)

In [ ]:
import rasterio.windows

# Read a 128×128-pixel tile from the centre of the image
with rasterio.open(raw_cog_path) as src:
    h, w = src.height, src.width

window = rasterio.windows.Window(
    col_off=w // 2 - 64, row_off=h // 2 - 64, width=128, height=128
)

rgba_tile = read_cog_tile_as_rgba(raw_cog_path, cmap="plasma", vmin=0, vmax=70, window=window)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(read_cog_tile_as_rgba(raw_cog_path, cmap="plasma", vmin=0, vmax=70), origin="upper")
axes[0].add_patch(plt.Rectangle((w//2-64, h//2-64), 128, 128, edgecolor="red", facecolor="none", lw=2))
axes[0].set_title("Full image with window highlighted")
axes[0].axis("off")

axes[1].imshow(rgba_tile, origin="upper")
axes[1].set_title(f"Window tile ({rgba_tile.shape[1]}×{rgba_tile.shape[0]} px)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 4. Remap an Existing COG to a New Colormap

`remap_cog_colormap` reads the raw float data from the source file, applies the
requested colormap, and writes a brand-new RGBA COG.  The original file is not
modified.

In [ ]:
remapped_paths = {}
for cmap_name in ["viridis", "hot", "grc_th", "jet"]:
    out_path = WORK_DIR / f"reflectivity_{cmap_name}.cog"
    remap_cog_colormap(
        raw_cog_path,
        out_path,
        new_cmap=cmap_name,
        # Use the vmin/vmax stored in the raw COG (no need to pass them again)
    )
    remapped_paths[cmap_name] = out_path
    meta = read_cog_metadata(out_path)
    print(f"{cmap_name:12s} → {out_path.name}  "
          f"(vmin={meta['vmin']}, vmax={meta['vmax']}, type={meta['data_type']})")

### 4.1 Custom vmin/vmax overrides (zooming the colour scale)

In [ ]:
out_narrow = WORK_DIR / "reflectivity_narrow_range.cog"
remap_cog_colormap(
    raw_cog_path,
    out_narrow,
    new_cmap="plasma",
    new_vmin=20.0,   # clip low-level echoes
    new_vmax=55.0,   # clip extreme values
)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
rgba_full  = read_cog_tile_as_rgba(raw_cog_path, cmap="plasma", vmin=0, vmax=70)
rgba_narrow = read_cog_tile_as_rgba(out_narrow)  # reads metadata from the file

axes[0].imshow(rgba_full, origin="upper")
axes[0].set_title("plasma  vmin=0, vmax=70")
axes[0].axis("off")

axes[1].imshow(rgba_narrow, origin="upper")
axes[1].set_title("plasma  vmin=20, vmax=55  (narrow range)")
axes[1].axis("off")

plt.suptitle("Effect of different vmin/vmax on the colour scale", fontsize=12, weight="bold")
plt.tight_layout()
plt.show()

### 4.2 Side-by-side comparison of all remapped COGs

In [ ]:
fig, axes = plt.subplots(1, len(remapped_paths), figsize=(16, 5))
for ax, (cmap_name, path) in zip(axes, remapped_paths.items()):
    rgba = read_cog_tile_as_rgba(path)  # uses metadata from the file
    ax.imshow(rgba, origin="upper")
    ax.set_title(cmap_name, fontsize=11)
    ax.axis("off")

plt.suptitle("Remapped RGBA COGs — each is a standalone file", fontsize=13, weight="bold")
plt.tight_layout()
plt.show()

## 5. Saving Rendered Tiles as PNG Files

You can easily save any tile as a PNG for serving via an API:

In [ ]:
# Export full-resolution overview 0 as PNG for each colormap
png_dir = WORK_DIR / "png_tiles"
png_dir.mkdir(exist_ok=True)

for cmap_name, cog_path in remapped_paths.items():
    rgba = read_cog_tile_as_rgba(cog_path, overview_level=0)
    img = Image.fromarray(rgba, mode="RGBA")
    png_path = png_dir / f"tile_{cmap_name}.png"
    img.save(png_path)
    print(f"Saved: {png_path}  ({rgba.shape[1]}×{rgba.shape[0]} px, {png_path.stat().st_size // 1024} KB)")

print(f"\nAll PNGs are in: {png_dir}")

## 6. Workflow Summary

```
Raw radar data (float, physical units)
        │
        ▼
create_raw_cog()          ──▶  raw_float COG  (stores vmin, vmax, default cmap)
        │
        ├──── read_cog_tile_as_rgba(cmap="viridis")  ──▶  RGBA numpy array  ──▶  PNG
        ├──── read_cog_tile_as_rgba(cmap="hot")      ──▶  RGBA numpy array  ──▶  PNG
        │
        └──── remap_cog_colormap(new_cmap="jet")     ──▶  RGBA COG (new file)
                      │
                      └──── read_cog_tile_as_rgba()  ──▶  RGBA numpy array  ──▶  PNG
```

**Key points:**

| Function | Input | Output | Modifies source? |
|---|---|---|---|
| `create_raw_cog` | float array + geometry | raw float COG | — |
| `create_cog` | float array + geometry | RGBA COG (pre-coloured) | — |
| `remap_cog_colormap` | raw float COG | new RGBA COG | ❌ No |
| `read_cog_tile_as_rgba` | any COG | RGBA numpy array | ❌ No |
| `read_cog_metadata` | any radarlib COG | dict of metadata | ❌ No |


In [ ]:
print("Done! All output files:")
for f in sorted(WORK_DIR.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(WORK_DIR)}  ({f.stat().st_size // 1024} KB)")